In [1]:
import pandas as pd
import numpy as np
import shutil

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

### text

In [3]:
df = pd.read_csv(raw_kyiv / "text_messages_sorted.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

In [4]:
chat_participants = (
    df.groupby(['donation_id', 'conversation_id'])['sender_id']
    .nunique()
    .reset_index(name='participants')
)

In [5]:
two_person_ids = chat_participants.loc[
    chat_participants['participants'] == 2,
    ['donation_id', 'conversation_id']
]
df_2p = df.merge(two_person_ids, on=['donation_id', 'conversation_id'])

In [6]:
def check_valid(group):
    total = len(group)
    max_share = group['sender_id'].value_counts(normalize=True).max()
    return total >= 100 and max_share <= 0.9

In [7]:
valid_mask = (
    df_2p.groupby(['donation_id', 'conversation_id'])
    .apply(check_valid)
)
valid_one_to_one = valid_mask[valid_mask].reset_index()[['donation_id', 'conversation_id']]

/var/folders/vx/kbkdd94d7092nh7qby2pf8lr0000gn/T/ipykernel_78587/1079480667.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_valid)


In [8]:
group_chats = chat_participants.loc[
    chat_participants['participants'] > 2,
    ['donation_id', 'conversation_id']
]

In [9]:
chats_to_keep = pd.concat([valid_one_to_one, group_chats]).drop_duplicates()
df = df.merge(chats_to_keep, on=['donation_id', 'conversation_id'])

In [10]:
donor_map = (
    df.groupby(["donation_id", "sender_id"])["conversation_id"]
    .nunique()
    .reset_index(name="n_conversations")
    .loc[lambda x: x.groupby("donation_id")["n_conversations"].transform('max') == x["n_conversations"]]
    .drop_duplicates("donation_id")  # на випадок ties
    .rename(columns={"sender_id": "donor_sender_id"})
    [["donation_id", "donor_sender_id"]]
)

In [11]:
df = (
    df.merge(donor_map, on="donation_id", how="left")
    .assign(is_donor=lambda x: (x["sender_id"] == x["donor_sender_id"]).astype(int))
    .sort_values(["donation_id", "conversation_id", "datetime"])
)

In [12]:
df.to_csv(
    filtered_data / "text_messages_filtered.csv",
    index=False
)

### audio

In [13]:
audio_df = pd.read_csv(raw_kyiv / "audio_messages_sorted.csv")
audio_df["datetime"] = pd.to_datetime(audio_df["datetime"])

In [14]:
text_df = pd.read_csv(filtered_data / "text_messages_filtered.csv")

In [15]:
donor_map = text_df[["donation_id", "donor_sender_id"]].drop_duplicates()

In [16]:
audio_df = (
    audio_df
    .merge(chats_to_keep, on=['donation_id', 'conversation_id']) 
    .merge(donor_map, on="donation_id", how="left")
    .assign(is_donor=lambda x: (x["sender_id"] == x["donor_sender_id"]).astype(int))
    .sort_values(["donation_id", "conversation_id", "datetime"])
    [["donation_id", "id", "conversation_id", "sender_id",
      "datetime", "length_seconds", "donor_sender_id", "is_donor"]]
)

In [17]:
audio_df.to_csv(
    filtered_data / "audio_messages_filtered.csv",
    index=False
)

In [18]:
for filename in ["comments_sorted.csv", "posts_sorted.csv", "reactions_sorted.csv"]:
    shutil.copy(raw_kyiv / filename, filtered_data / filename)